# Jukebox 1B Lyrics Quickstart

This reduced notebook targets modern Google Colab and focuses on the smallest useful milestone: install from your fork, load `1b_lyrics`, and generate one short top-level sample.

Before running the install cell, replace `your-github-user` with your GitHub username in `REPO_URL`.


In [0]:
REPO_URL = "https://github.com/your-github-user/jukebox.git"
!pip install git+{REPO_URL}

In [0]:
!nvidia-smi

In [0]:
import math
import torch as t
from IPython.display import Audio
from jukebox.make_models import make_prior, make_vqvae, MODELS
from jukebox.hparams import Hyperparams, setup_hparams
from jukebox.sample import _sample
from jukebox.utils.dist_utils import setup_dist_from_mpi

rank, local_rank, device = setup_dist_from_mpi()
device

The `1b_lyrics` top prior needs at least `6144` top-level tokens. With Jukebox's VQ-VAE, that corresponds to `786432` raw audio samples, or about `17.83` seconds at `44.1 kHz`. This quickstart uses that minimum valid length.


In [0]:
model = "1b_lyrics"
hps = Hyperparams()
hps.sr = 44100
hps.n_samples = 1
hps.name = "samples"
hps.levels = 3
hps.hop_fraction = [0.5, 0.5, 0.125]

vqvae_name, *prior_names = MODELS[model]
vqvae_hps = setup_hparams(vqvae_name, dict(sample_length=786432))
vqvae = make_vqvae(vqvae_hps, device)
top_prior = make_prior(setup_hparams(prior_names[-1], dict()), vqvae, device)
hps.sample_length = top_prior.n_ctx * top_prior.raw_to_tokens
hps.sample_length / hps.sr

In [0]:
metas = [dict(
    artist="Zac Brown Band",
    genre="Country",
    total_length=hps.sample_length,
    offset=0,
    lyrics="""I met a traveller from an antique land,
    Who said Two vast and trunkless legs of stone
    Stand in the desert. Near them, on the sand,
    Half sunk a shattered visage lies, whose frown,
    And wrinkled lip, and sneer of cold command,
    Tell that its sculptor well those passions read
    Which yet survive, stamped on these lifeless things,
    The hand that mocked them, and the heart that fed;
    And on the pedestal, these words appear:
    My name is Ozymandias, King of Kings;
    Look on my Works, ye Mighty, and despair!
    Nothing beside remains. Round the decay
    Of that colossal Wreck, boundless and bare
    The lone and level sands stretch far away
    """
)] * hps.n_samples
labels = [None, None, top_prior.labeller.get_batch_labels(metas, 'cuda')]
labels[-1]['y'].shape

In [0]:
sampling_temperature = 0.98
sampling_kwargs = [
    dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
    dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
    dict(temp=sampling_temperature, fp16=True, max_batch_size=16, chunk_size=32),
]
zs = [t.zeros(hps.n_samples, 0, dtype=t.long, device='cuda') for _ in range(len(prior_names))]
zs = _sample(zs, labels, sampling_kwargs, [None, None, top_prior], [2], hps)

In [0]:
Audio(f'{hps.name}/level_2/item_0.wav')